# Landing — Cost Entries

Extrai `finops_source.cost_entries` via psycopg2 e salva como **CSV** no MinIO.

- Sem Spark
- Watermark por `created_at`
- Particionado por `usage_date` (year/month/day)

In [ ]:
# parameters
execution_date = "2025-06-15"
year = 2025
month = 6
day = 15


In [ ]:
import os, io, csv, sys

notebooks_root = next(
    (p for p in ["/opt/airflow/notebooks", "/opt/spark/notebooks"]
     if os.path.exists(p)),
    "/opt/spark/notebooks",
)
if notebooks_root not in sys.path:
    sys.path.insert(0, notebooks_root)

import psycopg2, boto3
from utils.spark_session import get_watermark, update_watermark

# Conexão Postgres
def pg_conn():
    return psycopg2.connect(
        host=os.getenv("POSTGRES_HOST", "postgres"),
        port=int(os.getenv("POSTGRES_PORT", "5432")),
        dbname=os.getenv("POSTGRES_DB", "finops"),
        user=os.getenv("POSTGRES_USER", "finops_user"),
        password=os.getenv("POSTGRES_PASSWORD", "finops_pass"),
    )

# Cliente MinIO
def s3_client():
    return boto3.client(
        "s3",
        endpoint_url=os.getenv("MINIO_ENDPOINT", "http://minio:9000"),
        aws_access_key_id=os.getenv("MINIO_ACCESS_KEY", "minioadmin"),
        aws_secret_access_key=os.getenv("MINIO_SECRET_KEY", "minioadmin123"),
    )

def rows_to_csv(rows, columns):
    buf = io.StringIO()
    writer = csv.DictWriter(buf, fieldnames=columns)
    writer.writeheader()
    for row in rows:
        writer.writerow({col: (val.isoformat() if hasattr(val, "isoformat") else val)
                         for col, val in zip(columns, row)})
    return buf.getvalue()

BUCKET = os.getenv("MINIO_BUCKET", "datalake")


In [ ]:
ENTITY = "cost_entries"
watermark = get_watermark(ENTITY)
print(f"[landing_cost_entries] Watermark: {watermark}")

with pg_conn() as conn:
    with conn.cursor() as cur:
        cur.execute(
            "SELECT id, team_id, resource_name, resource_type, provider, region, "
            "       environment, usage_date, cost_usd, currency, tags, created_at "
            "FROM finops_source.cost_entries WHERE created_at > %s",
            (watermark,),
        )
        columns = [d[0] for d in cur.description]
        rows = cur.fetchall()

if not rows:
    print(f"[landing_cost_entries] Nenhum registro novo. Encerrando.")
else:
    # Agrupa por usage_date para manter particionamento por dia
    from collections import defaultdict
    by_date = defaultdict(list)
    for row in rows:
        usage_date = row[7]  # usage_date
        by_date[(usage_date.year, usage_date.month, usage_date.day)].append(row)

    s3 = s3_client()
    total = 0
    for (y, m, d), date_rows in by_date.items():
        csv_data = rows_to_csv(date_rows, columns)
        key = f"landing/cost_entries/cost_entries_{y:04d}-{m:02d}-{d:02d}.csv"
        s3.put_object(Bucket=BUCKET, Key=key, Body=csv_data.encode())
        total += len(date_rows)

    new_wm = max(r[-1] for r in rows)
    update_watermark(ENTITY, new_wm, row_count=total)
    print(f"[landing_cost_entries] {total} registros em {len(by_date)} partições")
    print(f"[landing_cost_entries] Watermark → {new_wm}")
